# Анализ: что делает рецепт хорошим?
## Совпадают ли популярность, польза и реакция?

Работаем на готовой мастер-таблице `data/recipes_master.csv` (собрана в `recipes_dataset.ipynb`).

**Гипотеза:** рейтинг рецепта почти не связан с его пользой — люди голосуют за вкус, а не за здоровье.

In [1]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path("data") / "recipes_master.csv")
print("Всего рецептов:", len(df))

# Надёжные рейтинги: оставляем рецепты хотя бы с 10 оценками,
# иначе средний рейтинг по 1-2 голосам — шум.
rel = df[df["n_ratings"] >= 10].copy()
print("С >= 10 оценками:", len(rel))
rel[["name", "avg_rating", "n_ratings", "calories", "sugar_pdv"]].head()

Всего рецептов: 231637
С >= 10 оценками: 21399


,name,avg_rating,n_ratings,calories,sugar_pdv
12,better then bush s baked beans,3.900000,10,462.4,214.0
15,chicken lickin good pork chops,4.368421,19,105.7,0.0
16,chile rellenos,4.045455,22,94.0,0.0
17,chinese candy,4.833333,12,232.7,77.0
33,grilled venison burgers,4.428571,14,190.9,10.0


## 1. Связан ли рейтинг с пользой? (корреляции)

Корреляция близкая к 0 означает «связи нет». Смотрим, тянет ли средний рейтинг за калориями, сахаром, жирами и т.д.

In [2]:
health_cols = ["calories", "sugar_pdv", "sat_fat_pdv", "protein_pdv", "sodium_pdv"]
corr = rel[["avg_rating"] + health_cols].corr()["avg_rating"].drop("avg_rating")
print("Корреляция avg_rating с показателями пользы:")
print(corr.round(3))

# Вывод: все значения около нуля -> рейтинг ПРАКТИЧЕСКИ НЕ ЗАВИСИТ от пользы.

Корреляция avg_rating с показателями пользы:
calories      -0.030
sugar_pdv     -0.039
sat_fat_pdv   -0.018
protein_pdv   -0.012
sodium_pdv    -0.013
Name: avg_rating, dtype: float64


## 2. Контраст: любимое — но «вредное»

Берём рецепты, которые люди обожают (рейтинг ≥ 4.5 при ≥ 50 оценках), и сортируем по сахару. Если гипотеза верна — в топе будут совсем не здоровые блюда.

In [3]:
loved = rel[(rel["avg_rating"] >= 4.5) & (rel["n_ratings"] >= 50)]
print("Любимых рецептов (>=4.5, >=50 оценок):", len(loved))

cols = ["name", "avg_rating", "n_ratings", "sugar_pdv", "calories"]
loved.sort_values("sugar_pdv", ascending=False)[cols].head(10)

Любимых рецептов (>=4.5, >=50 оценок): 1063


,name,avg_rating,n_ratings,sugar_pdv,calories
132355,mean chef s maple brine,4.760000,50,2508.0,3147.7
132364,mean chef s pineapple upside down cake,4.766667,60,2154.0,5496.2
151671,paige s buttercream frosting,4.924528,53,1880.0,3536.2
22149,best apple pie jam,4.688525,61,1419.0,1468.1
61349,cream cheese almond coffee cake,4.822581,62,1325.0,4208.5
54076,cinnamon swirl quick bread,4.788571,175,1115.0,2539.8
16141,banana banana nut bread,4.892308,65,1037.0,3093.3
22900,best ever banana bread honestly,4.603448,58,999.0,3107.8
18441,basic pancake syrup,4.560976,82,991.0,1252.7
118069,kittencal s easy creamy white glaze,4.875000,64,942.0,1157.0


## 3. Четыре квадранта: вкусно/невкусно × полезно/вредно

Разобьём рецепты на группы по медианам рейтинга и сахара и посмотрим, куда попадает большинство «любимых».

In [4]:
tasty = rel["avg_rating"] >= rel["avg_rating"].median()
sweet = rel["sugar_pdv"] >= rel["sugar_pdv"].median()

rel["quadrant"] = (
    tasty.map({True: "любят", False: "не любят"})
    + " / "
    + sweet.map({True: "много сахара", False: "мало сахара"})
)
rel["quadrant"].value_counts()

quadrant
не любят / много сахара    5567
любят / много сахара       5355
любят / мало сахара        5347
не любят / мало сахара     5130
Name: count, dtype: int64

## 4. Рецепты-долгожители vs однодневки

Другой угол — не польза, а **жизнь рецепта во времени**. По датам отзывов (`date`) смотрим размах между первым и последним откликом: одни собирают отзывы годами, другие вспыхивают и гаснут. Данные охватывают 2000–2018.

In [5]:
# Размах жизни рецепта = разница между первым и последним отзывом
inter = pd.read_csv(
    Path("data") / "RAW_interactions.csv",
    usecols=["recipe_id", "date", "rating"],
    parse_dates=["date"],
)

life = inter.groupby("recipe_id").agg(
    n_ratings=("rating", "size"),
    avg_rating=("rating", "mean"),
    first_review=("date", "min"),
    last_review=("date", "max"),
)
life["span_days"] = (life["last_review"] - life["first_review"]).dt.days
life["span_years"] = (life["span_days"] / 365.25).round(1)
life["name"] = life.index.map(df.set_index("id")["name"])

print("Период данных:", inter["date"].min().date(), "->", inter["date"].max().date())
life[["name", "n_ratings", "span_years"]].head()

Период данных: 2000-01-25 -> 2018-12-20


,name,n_ratings,span_years
recipe_id,,,
38,low fat berry blue frozen dessert,4,6.5
39,biryani,1,0.0
40,best lemonade,9,7.4
41,carina s tofu vegetable kebabs,2,5.8
43,best blackbottom pie,1,0.0


### Долгожители

Много отзывов **и** растянуты на годы — «вечные» рецепты, которые готовят десятилетиями.

In [6]:
pop = life[life["n_ratings"] >= 30]   # хотя бы 30 отзывов, чтобы размах был осмысленным
pop.sort_values("span_years", ascending=False)[
    ["name", "n_ratings", "avg_rating", "span_years"]
].head(10)

,name,n_ratings,avg_rating,span_years
recipe_id,,,,
13307,neiman marcus 250 chocolate chip cookies recipe,148,4.277027,18.5
3603,hooters buffalo wings,161,4.801242,18.0
607,famous barr s french onion soup,147,4.074830,18.0
7435,kevin s best corned beef,41,4.341463,18.0
3470,crock pot chicken gravy and stuffing,322,4.397516,17.9
4627,chicken tortilla soup ii,591,4.553299,17.9
5170,pete s scratch pancakes,672,4.389881,17.8
8531,crock pot beef stew,64,4.109375,17.8
4423,jiffy corn pudding,37,4.702703,17.7


### Однодневки

Десятки отзывов, но всё уложилось в пару недель — вспыхнули и забылись.

> Любопытный паттерн: у многих суффиксы `sp5`, `5fix`, `rsc`, `ragu` — это **конкурсные/спонсорские** рецепты. Их активно отзывали в период челленджа, а потом забросили.

In [7]:
flash = pop[pop["span_days"] < 60]
print("однодневок (>=30 отзывов, но всё за <60 дней):", len(flash))
flash.sort_values("n_ratings", ascending=False)[
    ["name", "n_ratings", "avg_rating", "span_days"]
].head(10)

однодневок (>=30 отзывов, но всё за <60 дней): 23


,name,n_ratings,avg_rating,span_days
recipe_id,,,,
515167,simply irresistible tropical potato salad sp5,74,4.932432,7
495577,shredded potato baskets with cheese and bacon ...,73,4.232877,39
514605,animal style skillet potatoes sp5,53,4.339623,9
524863,ragu shuka ragu,50,4.900000,10
495124,warm roasted root vegetable and chicken salad ...,47,4.893617,37
475446,shrimp nicoise quiche,47,4.255319,34
514423,mac n cheese and spinach strata sp5,44,4.545455,20
487111,back porch bayou shrimp corn rsc,42,4.761905,23
494631,grilled potatoes shrimp with spinach mousse ...,41,4.512195,23


## 5. Правда ли «здоровые» рецепты здоровые?

В `tags` есть метки `low-fat`, `healthy`, `low-calorie` и т.п. Сравним реальные нутриенты рецептов **с** таким тегом и **без** него. Колонки `Δ%` — на сколько процентов медиана у тегированных отличается от остальных (отрицательное = ниже).

**Гипотеза:** `low-fat` режут жир, но компенсируют сахаром.

In [8]:
# Для каждого тега сравниваем медианы нутриентов: рецепты С тегом vs БЕЗ.
# Кавычки в "'low-fat'" — чтобы матчить тег целиком, а не как подстроку.
health_tags = ["low-fat", "healthy", "low-calorie", "low-carb", "low-cholesterol", "low-sodium"]
metrics = ["calories", "total_fat_pdv", "sugar_pdv", "sodium_pdv"]

rows = []
for tag in health_tags:
    has = df["tags"].str.contains(f"'{tag}'", regex=False, na=False)
    row = {"tag": tag, "n_recipes": int(has.sum())}
    for m in metrics:
        with_t = df.loc[has, m].median()
        without_t = df.loc[~has, m].median()
        row[f"{m} Δ%"] = round((with_t - without_t) / without_t * 100)
    rows.append(row)

# Колонки Δ% : насколько медиана у тегированных отличается от остальных (− = ниже)
pd.DataFrame(rows).set_index("tag")

,n_recipes,calories Δ%,total_fat_pdv Δ%,sugar_pdv Δ%,sodium_pdv Δ%
tag,,,,,
low-fat,22170,-34,-87,54,-47
healthy,40340,-26,-71,52,-33
low-calorie,36429,-32,-52,-4,-13
low-carb,42189,-3,32,-48,46
low-cholesterol,36743,-31,-65,21,-33
low-sodium,43349,-9,-29,57,-79


### Вывод

Каждый тег честен ровно по «своему» показателю, но **компенсирует другим**:

- **low-fat**: жир −87%, но **сахар +54%** — классическая компенсация.
- **healthy**: жир −71%, сахар +52%.
- **low-sodium**: натрий −79%, но **сахар +57%**.
- **low-cholesterol**: жир −65%, сахар +21%.
- **low-carb** — зеркально: сахар −48%, зато жир +32% и натрий +46% (кето-логика).
- **low-calorie** — самый честный: калории −32%, сахар почти не меняется (−4%).

Итог: «здоровый» тег ≠ здоровый рецепт целиком — важно, что именно он оптимизирует.

## 6. Рецепты, которые врут про время

Сравниваем заявленное `minutes` с числом шагов `n_steps`. Метрика **«минут на шаг»** = `minutes / n_steps`: если рецепт обещает 15 минут при 25 шагах, на шаг приходится меньше минуты — приготовить за обещанное время нереально.

In [10]:
# Время на шаг = minutes / n_steps. Сначала чистим выбросы:
# в Food.com есть minutes=0 и абсурдные значения (до 2.1 млрд минут).
clean = df[df["minutes"].between(1, 600) & (df["n_steps"] >= 1)].copy()
clean["min_per_step"] = clean["minutes"] / clean["n_steps"]
print("Медиана 'минут на шаг' по всем рецептам:", round(clean["min_per_step"].median(), 2))

# «Обман»: обещают <= 15 минут, но >= 20 шагов
liars = clean[(clean["minutes"] <= 15) & (clean["n_steps"] >= 20)]
print("Рецептов-обманщиков (<=15 мин, >=20 шагов):", len(liars))

liars.sort_values("min_per_step")[
    ["name", "minutes", "n_steps", "min_per_step"]
].head(10)

Медиана 'минут на шаг' по всем рецептам: 4.09
Рецептов-обманщиков (<=15 мин, >=20 шагов): 270


,name,minutes,n_steps,min_per_step
122836,lemons lots of great uses for them,1,49,0.020408
178877,salmon and rice wrapped in pastry with dill sauce,1,39,0.025641
20621,beef shreds with green pepper,1,28,0.035714
42633,chicken and cashew curry with chutney and mint...,2,54,0.037037
226114,white chocolate and raspberry cheesecake,2,37,0.054054
107958,honey almond cookies,2,35,0.057143
125346,lobster thermidor a la julia child,2,32,0.062500
38347,champagne 101,2,30,0.066667
82010,fancy shmancy dinner crepes,2,27,0.074074
103583,healthy instant ersatz cheesecake,2,27,0.074074


### Вывод

Типичный рецепт даёт ~**4 минуты на шаг**. «Обманщики» обещают доли минуты на шаг — это физически нереально (нельзя выполнить 73 шага за 15 минут).

⚠️ Оговорка: часть «обманщиков» — не блюда, а **сборники-гайды** («lots of great uses», «variations»): у них много пунктов, но это не последовательные шаги готовки. Для чистого результата их стоит отфильтровать по названию/тегам.

## 7. Сезонность готовки

Из даты отзыва (`date`) достаём **месяц** (`.dt.month`) и смотрим, какие сезонные теги всплывают когда. Считаем не абсолютные числа, а **долю** отзывов месяца — иначе результат исказит общий рост популярности сайта со временем.

In [11]:
# Чистый datetime: вытаскиваем месяц из даты отзыва
inter = pd.read_csv(
    Path("data") / "RAW_interactions.csv",
    usecols=["recipe_id", "date"],
    parse_dates=["date"],
)
inter["month"] = inter["date"].dt.month   # .dt даёт доступ к компонентам даты

# привязываем теги рецепта к каждому отзыву
inter["tags"] = inter["recipe_id"].map(df.set_index("id")["tags"])
inter = inter.dropna(subset=["tags"])

# доля отзывов с сезонным тегом в каждом месяце (% от всех отзывов месяца)
total_by_month = inter.groupby("month").size()
seasonal = ["christmas", "cookies-and-brownies", "thanksgiving", "soups-stews",
            "salads", "summer", "grilling", "frozen-desserts"]

share = {}
for tag in seasonal:
    has = inter["tags"].str.contains(f"'{tag}'", regex=False, na=False)
    monthly = inter[has].groupby("month").size().reindex(range(1, 13), fill_value=0)
    share[tag] = (monthly / total_by_month * 100).round(1)

month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
season_df = pd.DataFrame(share)
season_df.index = month_names
season_df   # строки = месяцы, значения = % отзывов месяца с этим тегом

,christmas,cookies-and-brownies,thanksgiving,soups-stews,salads,summer,grilling,frozen-desserts
Jan,5.1,5.6,2.7,7.2,3.9,2.9,1.6,0.4
Feb,3.8,5.8,2.0,6.1,4.1,3.0,1.7,0.4
Mar,3.3,5.2,1.8,5.1,4.7,3.3,1.9,0.4
Apr,3.3,5.3,1.9,4.2,5.7,4.0,2.4,0.6
May,2.8,5.3,1.5,3.4,6.8,5.5,3.4,0.7
Jun,2.5,4.5,1.4,2.8,8.2,6.7,4.1,1.0
Jul,2.7,4.7,1.3,2.8,8.2,7.6,4.2,1.1
Aug,2.6,4.6,1.5,3.3,6.7,6.7,3.4,0.9
Sep,3.1,5.3,2.0,4.7,5.2,4.9,2.6,0.5
Oct,3.9,5.8,3.2,6.5,4.0,3.5,1.8,0.4


### Вывод

Готовка явно **сезонна** — доля отзывов с тегом меняется в разы по месяцам:

- ❄️ **Декабрь:** `christmas` 13.6% и `cookies-and-brownies` 13.5% (против ~3–4% летом) — праздничная выпечка.
- 🥗 **Лето (Jun–Jul):** `salads` (Jun 8.2%), `grilling` + `summer` (Jul), `frozen-desserts` — лёгкое и на гриле.
- 🍲 **Январь:** `soups-stews` 7.2% — согревающее после праздников.

Люди готовят **по сезону**, даже спустя годы после публикации рецепта — отзывы это фиксируют.

## 8. Честны ли «здоровые» теги? Проверка через USDA

Идея: только USDA даёт **точные граммы**, чтобы доказать, что ингредиент — это сахар.

**Ограничение метода:** в Food.com у ингредиентов нет количеств (только названия), поэтому пересчитать точные граммы нутриентов на рецепт нельзя. Зато можно поймать **подмену**: USDA подтверждает, что `sugar`/`honey`/`syrup` = 60–99 г сахара на 100 г, а дальше считаем, как часто «здоровые» рецепты их содержат по сравнению со всеми.

In [13]:
import ast

# Подсластители, ПОДТВЕРЖДЁННЫЕ USDA (sugar г на 100 г, SR Legacy):
#   white sugar 99.2 | powdered 97.8 | brown 97.0 | honey 82.1 | corn syrup 77.6 | maple 60.5
# => наличие любого из них в составе = "добавленный сахар".
USDA_SWEETENER_SUGAR = {       # для справки/обоснования
    "white sugar": 99.2, "brown sugar": 97.0, "powdered sugar": 97.8,
    "honey": 82.1, "corn syrup": 77.6, "maple syrup": 60.5,
}
SWEET_WORDS = ["sugar", "honey", "syrup", "molasses", "agave"]

def n_sweeteners(ingredients_str):
    items = ast.literal_eval(ingredients_str)
    return sum(any(w in ing.lower() for w in SWEET_WORDS) for ing in items)

df["n_sweet"] = df["ingredients"].apply(n_sweeteners)
df["has_sweet"] = df["n_sweet"] > 0

rows = []
for tag in ["low-fat", "healthy", "low-calorie", "low-carb"]:
    has = df["tags"].str.contains(f"'{tag}'", regex=False, na=False)
    rows.append({"group": tag, "n_recipes": int(has.sum()),
                 "% со сладким": round(df.loc[has, "has_sweet"].mean() * 100, 1),
                 "ср. подсласт.": round(df.loc[has, "n_sweet"].mean(), 2)})
rows.append({"group": "ВСЕ рецепты", "n_recipes": len(df),
             "% со сладким": round(df["has_sweet"].mean() * 100, 1),
             "ср. подсласт.": round(df["n_sweet"].mean(), 2)})
pd.DataFrame(rows).set_index("group")

,n_recipes,% со сладким,ср. подсласт.
group,,,
low-fat,22170,42.3,0.48
healthy,40340,45.5,0.55
low-calorie,36429,26.8,0.29
low-carb,42189,18.0,0.19
ВСЕ рецепты,231637,39.4,0.50


### Вывод

- **healthy**: 45.5% рецептов содержат подсластитель — заметно выше среднего (39.4%). Подмена видна.
- **low-fat**: 42.3% — тоже выше среднего, хоть и умереннее.
- **low-calorie** (26.8%) и **low-carb** (18.0%) — наоборот, реже среднего: эти теги сахар не добавляют.

USDA здесь дал **числовое основание** называть эти ингредиенты сахаром (60–99 г/100 г). Вместе с разделом 5 (где у low-fat сахар +54% по %DV) картина сходится: ярлыки `healthy`/`low-fat` действительно чаще опираются на сладкое.

⚠️ **Честное ограничение:** в Food.com у ингредиентов нет количеств (только названия), поэтому пересчитать точные граммы на рецепт через USDA нельзя. Мы ловим подмену по *наличию* подсластителей, а не по их массе.

## Что дальше

1. **Тексты отзывов** (`review` из `RAW_interactions.csv`): за что хвалят — за вкус/простоту или за пользу? Частоты слов / тональность.
2. **Время и сложность** (`minutes`, `n_steps`): любят ли простые рецепты больше?
3. **USDA-перепроверка** калорий/сахара по ингредиентам (см. Шаг 5 в `recipes_dataset.ipynb`).

# 🏁 Выводы: что делает рецепт «хорошим»?

**Главный вопрос проекта:** совпадают ли три «правды» о рецепте — популярность (рейтинг), польза (нутриенты) и реакция/контекст (отзывы, теги, время)?

**Короткий ответ: нет. Три правды расходятся, а ярлыки систематически вводят в заблуждение.**

| # | Находка | Доказательство |
|---|---------|----------------|
| 1 | Рейтинг **не связан** с пользой | корреляция rating ↔ калории/сахар/жир ≈ **0** (−0.01…−0.04) |
| 2 | Любимое — часто **вредное** | ★4.92 у buttercream frosting при сахаре 1880% нормы |
| 3 | Вкус и сахар **независимы** | «любимые» рецепты делятся 50/50 на сладкие и нет |
| 4 | Популярность бывает **искусственной** | долгожители живут 18 лет vs конкурсные однодневки — 7 дней |
| 5 | «Здоровые» теги **компенсируют** | `low-fat`: жир −87%, но **сахар +54%** |
| 6 | Рецепты **врут про время** | при медиане 4 мин/шаг есть «15 мин / 73 шага» |
| 7 | Готовка **сезонна** | `christmas` — пик 13.6% в декабре vs 2.5% летом |
| 8 | USDA подтверждает **подмену сахаром** | `healthy`/`low-fat` чаще содержат подсластители (60–99 г/100 г) |

## Главная мысль

«Хороший рейтинг» означает **«вкусно»**, а не «полезно», «быстро» или «честно». Чем громче ярлык — `healthy`, `low-fat`, `15-minutes`, верхушка топа — тем чаще он скрывает компромисс: сахар вместо жира, партия вместо порции, всплеск вместо любви. **Три правды о рецепте живут отдельно друг от друга** — и в этом главный, неинтуитивный вывод.

## Честные ограничения

- **Нет количеств ингредиентов** → точные граммы на рецепт по USDA не пересчитать; ловим подмену только по *наличию* подсластителей.
- **Нутриенты Food.com — на весь рецепт, не на 100 г** → топ по сахару смещён к «большим» заготовкам (глазури, маринады, джемы).
- **Матчинг USDA по строке ненадёжен** (`salt`→butter) → USDA применяли точечно, только для однозначных подсластителей.
- **Псевдо-рецепты** (гайды вроде «lemons — lots of uses») зашумляют разделы о тегах и времени.